In [2]:
with open("Sherlock_Holmes.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 3868122




                          THE COMPLETE SHERLOCK HOLMES

                               Arthur C


In [3]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [5]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [12]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|-|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['THE', 'COMPLETE', 'SHERLOCK', 'HOLMES', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'A', 'Study', 'In', 'Scarlet', 'The', 'Sign', 'of', 'the', 'Four', 'The', 'Adventures', 'of', 'Sherlock', 'Holmes', 'A', 'Scandal', 'in', 'Bohemia', 'The', 'Red']


In [13]:
print(len(preprocessed))

807987


### Next we sort the vocabulary in alphabetical order and assign them token IDs

In [14]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words) 
print("Vocabulary size:", vocab_size)

Vocabulary size: 21194


In [15]:
vocab = {token:integer for integer,token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

('!', 0)
('"', 1)
('&', 2)
("'", 3)
('(', 4)
(')', 5)
('*1', 6)
(',', 7)
('-', 8)
('--', 9)
('.', 10)
('//sherlock', 11)
('000', 12)
('1', 13)
('10', 14)
('100', 15)
('1000', 16)
('104', 17)
('109', 18)
('10s', 19)
('10th', 20)


In [43]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|-|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations- Eg:"Hello , world !" -> "Hello, world!"
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [44]:
tokenizer = SimpleTokenizerV1(vocab)

text = """So alarming did the state of my finances become, that I soon realized that I must either leave the metropolis and rusticate somewhere in the country, or that I must make a complete alteration in my style of living."""
ids = tokenizer.encode(text)
print(ids)

[3587, 4694, 8505, 19247, 18406, 14336, 13956, 10098, 5521, 7, 19244, 2021, 18083, 16155, 19244, 2021, 13939, 9202, 12855, 19247, 13596, 4840, 16981, 18076, 11910, 19247, 7680, 7, 14440, 19244, 2021, 13939, 13289, 4328, 7211, 4756, 11910, 13956, 18695, 14336, 13065, 10]


In [45]:
tokenizer.decode(ids)

'So alarming did the state of my finances become, that I soon realized that I must either leave the metropolis and rusticate somewhere in the country, or that I must make a complete alteration in my style of living.'

In [46]:
text1 = """ "That's a strange thing," remarked my companion; "you are the second man to-day that has used that expression to me." """
ids1 = tokenizer.encode(text1)
print(ids1)

[1, 3849, 3, 16989, 4328, 18579, 19288, 7, 1, 16436, 13956, 7178, 191, 1, 21143, 5021, 19247, 17277, 13309, 19456, 8, 8081, 19244, 11289, 20269, 19244, 9751, 19456, 13464, 10, 1]


In [47]:
tokenizer.decode(ids1)

'" That\' s a strange thing," remarked my companion ;" you are the second man to - day that has used that expression to me."'

### Unknown or non-encoded words introduced:

In [48]:
text2 = "Hello, do you like tea?"
ids2 = tokenizer.encode(text2)
print(ids2)

KeyError: 'Hello'

In [49]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [50]:
len(vocab.items())

21196

In [51]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('£88', 21191)
('à', 21192)
('écarté', 21193)
('<|endoftext|>', 21194)
('<|unk|>', 21195)


##### LLM's are trained using unsupervised/self supervised method and it is autoregressive(O/P of previous input used in next step in next word/token prediction)

In [52]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [53]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [54]:
tokenizer.encode(text)

[21195,
 7,
 8820,
 21143,
 12984,
 19118,
 192,
 21194,
 2045,
 19247,
 18821,
 21195,
 14336,
 19247,
 14667,
 10]

In [55]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit <|unk|> of the palace.'

"Hello" and "terraces" are unknown tokens/words in the vocabulary.

In [8]:
import re
text = "This costs $5.99, which is quite expensive!"
res = re.sub(r'(\$)(\d.\d+)', r'\2 USD', text)
print(res)
name = "Thomas, Mark Stephen"
res1 = re.sub(r'(\w+),\s+(.*)', r'\2 \1', name)
print(res1)

This costs 5.99 USD, which is quite expensive!
Mark Stephen Thomas


### Byte Pair Encoding(BPE):
ensures most common words in vocab are represented as a single token while rare words are broken down into two or more subword tokens. BPE looks into frequency of a particular sequence, subtracts its total occurences from the original and if it becomes 0, it removes it from the frequency table. Like such it looks at tokens present and forms a vocab of tokens eventually having good enough subwords that cover almost every word.

In [10]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [11]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [12]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [31]:
# Unknown tokens
integers = tokenizer.encode("Akwirw ier")
print(integers)

strings = tokenizer.decode(integers)
print(strings)

integers1 = tokenizer.encode("aaabdaaabac")
print(integers1)

strings1 = tokenizer.decode(integers1)
print(strings1)

[33901, 86, 343, 86, 220, 959]
Akwirw ier
[7252, 397, 6814, 64, 397, 330]
aaabdaaabac


In [14]:
with open("Sherlock_Holmes.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

1131297


In [20]:
enc_sample = enc_text[20000:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y: {y}")

x: [587, 612, 1141, 262]
y: [612, 1141, 262, 1755]


In [21]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[587] ----> 612
[587, 612] ----> 1141
[587, 612, 1141] ----> 262
[587, 612, 1141, 262] ----> 1755


In [22]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 been ---->  there
 been there ---->  during
 been there during ---->  the
 been there during the ---->  night


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    # Initialize tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

##### Generally for huge models max_length = stride, so that same tokens are processed multiple times. Therefore saves time and compute power. We shuffle the batches and not the tokens itself, so the sentence structure is still kept but the model doesn't learn bad correlaions. Eg: In wikipedia if two articles(one comes right after another) model might make an assumption that they are related and that that order is to be preserved.

In [25]:
with open("Sherlock_Holmes.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [26]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[628, 628, 220, 220]]), tensor([[628, 220, 220, 220]])]


In [27]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[628, 220, 220, 220]]), tensor([[220, 220, 220, 220]])]


In [28]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  628,   628,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,  3336],
        [49269,  9328,  6006,  1137]])

Targets:
 tensor([[  628,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,   220,   220],
        [  220,   220,  3336, 49269],
        [ 9328,  6006,  1137, 36840]])


### Create Token Embeddings

In [29]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [30]:
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)
